Import des données depuis Kaggle

In [2]:
from copy import deepcopy

import kagglehub
import os

path = kagglehub.dataset_download("ashery/chexpert")
print("Chemin local :", path)
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

Chemin local : /Users/arnaudbernard/.cache/kagglehub/datasets/ashery/chexpert/versions/1
/Users/arnaudbernard/.cache/kagglehub/datasets/ashery/chexpert/versions/1/train.csv


Création d'un DataFrame avec les données (code venant de Kaggle)

In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "train.csv"  # ou le chemin relatif exact retourné à l'étape 1

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "ashery/chexpert",
    file_path,
)
print(df.head())

/var/folders/x4/5gzk8qw96k94xqgxgfjbl5v00000gn/T/ipykernel_88218/4215216645.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


                                                Path     Sex  Age  \
0  CheXpert-v1.0-small/train/patient00001/study1/...  Female   68   
1  CheXpert-v1.0-small/train/patient00002/study2/...  Female   87   
2  CheXpert-v1.0-small/train/patient00002/study1/...  Female   83   
3  CheXpert-v1.0-small/train/patient00002/study1/...  Female   83   
4  CheXpert-v1.0-small/train/patient00003/study1/...    Male   41   

  Frontal/Lateral AP/PA  No Finding  Enlarged Cardiomediastinum  Cardiomegaly  \
0         Frontal    AP         1.0                         NaN           NaN   
1         Frontal    AP         NaN                         NaN          -1.0   
2         Frontal    AP         NaN                         NaN           NaN   
3         Lateral   NaN         NaN                         NaN           NaN   
4         Frontal    AP         NaN                         NaN           NaN   

   Lung Opacity  Lung Lesion  Edema  Consolidation  Pneumonia  Atelectasis  \
0           NaN     

Import des données sous forme d'un DataFrame et premier nettoyage pour avoir **uniquement des valeurs numérique**.
Les colonnes concernées sont ```Sex``` (Male, Female -> 0, 1) et ```Frontal/Latéral``` (Frontal, Latérale -> 0, 1)

In [4]:
import pandas as pd
import copy

def numeric_df(df):
    new_df = copy.deepcopy(df)

    sexes = {"Male": 0, "Female": 1}
    new_df["Sex"] = new_df["Sex"].map(sexes)

    axes = {"Frontal": 0, "Lateral": 1}
    new_df["Frontal/Lateral"] = new_df["Frontal/Lateral"].map(axes)

    new_df = pd.get_dummies(new_df, columns=["AP/PA"], dtype=int)
    new_df = new_df.rename(columns={"AP/PA_AP": "AP", "AP/PA_PA": "PA"})

    return new_df

for root, dirs, files in os.walk("data"):
    for file in files:
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(root, file))

            df = numeric_df(df)

            os.makedirs("cleaned", exist_ok=True)
            name_file, ext = os.path.splitext(file)
            df.to_csv(os.path.join("cleaned", f"{name_file}_cleaned{ext}"))

Premier nettoyage des images ```.jpg``` en recadrant sur le centre avec une **taille de 320x320**.

In [6]:
from PIL import Image

SIZE = 320


def crop_img(image):
    img_w, img_h = image.size

    left = 0 + (img_w - SIZE) // 2
    top = 0 + (img_h - SIZE) // 2
    right = left + SIZE
    bottom = top + SIZE

    img_cropped = image.crop((left, top, right, bottom))
    return img_cropped

input_root = "data"
output_root = "cleaned"

for root, dirs, files in os.walk("data"):
    for file in files:
        if file.endswith(".jpg") and not file.startswith("._"):
            img = Image.open(os.path.join(root, file)).convert('L')

            img_cropped = crop_img(img)

            rel_path = os.path.relpath(root, input_root)    # Chemin relatif

            # Recréer les dossiers dans ./cleaned
            output_dir = os.path.join(output_root, rel_path)
            os.makedirs(output_dir, exist_ok=True)

            name_file, ext = os.path.splitext(file)
            img_cropped.save(os.path.join(output_dir, f"{name_file}_cleaned{ext}"))
